# M2 - Frozen, versioned snapshot (track C)

Same features as `01_snapshot_to_datastore`, but the output is **frozen and self-describing**:

```
ml/training/site=<site>/snapshot=<YYYYMMDD-HHMM>/
  data.parquet
  _manifest.json      <- written LAST: its presence means the snapshot is complete
```

- **Fixed input**: the Delta table is read at one explicit version (`versionAsOf`), recorded in the manifest as `source_delta_version`.
- **Write-once**: if `snapshot=<id>/` already exists, the notebook reports it and stops. It never overwrites and never bumps silently.
- **Canonical manifest**: sorted keys, UTF-8; its SHA-256 is taken over the bytes written, so `prep` and `scripts/72_register_snapshot_asset.py` recompute the same hash.

Auth: no keys. `notebookutils.fs.mount` uses the Entra token of the user running the notebook (Storage Blob Data Contributor on `<STORAGE_ACCOUNT>`).

`01_snapshot_to_datastore` is untouched so track A stays replayable. The feature code is duplicated on purpose: a Fabric notebook cannot import from the repo.

In [ ]:
# --- Parameters -------------------------------------------------------------
import re
from datetime import datetime, timezone

WORKSPACE_ID = '<FABRIC_WORKSPACE_ID>'
KQL_DATABASE_ID = '<KQL_DATABASE_ID>'
TABLE_NAME = 'telemetry'

STORAGE_ACCOUNT = '<STORAGE_ACCOUNT>'
FILESYSTEM = 'snapshots'
SITE = 'demo'

# Leave empty to derive the id from the current UTC time (YYYYMMDD-HHMM).
# To REPLAY a run (and see the write-once guard answer), set it explicitly.
SNAPSHOT_ID = ''

HORIZON_MIN = 15
WINDOW_MIN = 30
LAGS_MIN = [1, 5, 15]
SENSORS = ['temperature_c', 'humidity_pct', 'pressure_bar', 'vibration_mm_s']
TARGET = 'target_temperature_c'

MIN_USABLE_MIN_PER_DEVICE = 120
RECOMMENDED_MIN_PER_DEVICE = 480

DATA_FILE = 'data.parquet'
MANIFEST_FILE = '_manifest.json'

if not SNAPSHOT_ID:
    SNAPSHOT_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')
if not re.fullmatch('[0-9]{8}-[0-9]{4}', SNAPSHOT_ID):
    raise ValueError(f'SNAPSHOT_ID must look like YYYYMMDD-HHMM, got {SNAPSHOT_ID!r}')

ONELAKE_PATH = (
    f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com'
    f'/{KQL_DATABASE_ID}/Tables/{TABLE_NAME}'
)
MOUNT_POINT = '/snapshots'
SNAPSHOT_PREFIX = f'ml/training/site={SITE}/snapshot={SNAPSHOT_ID}'

print(f'snapshot id : {SNAPSHOT_ID}')
print(f'source      : {ONELAKE_PATH}')
print(f'destination : abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{SNAPSHOT_PREFIX}')

## 1. Write-once guard (before any computation)

If the target folder exists, stop here. A folder without `_manifest.json` is an interrupted run: it is reported as incomplete and must be cleaned up by a human, never completed silently.

In [ ]:
import os

import notebookutils

abfss_root = f'abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net'

if any(m.mountPoint == MOUNT_POINT for m in notebookutils.fs.mounts()):
    notebookutils.fs.unmount(MOUNT_POINT)
notebookutils.fs.mount(abfss_root, MOUNT_POINT)

local_root = notebookutils.fs.getMountPath(MOUNT_POINT)
snapshot_dir = os.path.join(local_root, *SNAPSHOT_PREFIX.split('/'))
manifest_path = os.path.join(snapshot_dir, MANIFEST_FILE)
data_path = os.path.join(snapshot_dir, DATA_FILE)

if os.path.exists(snapshot_dir):
    complete = os.path.isfile(manifest_path)
    state = 'COMPLETE (manifest present)' if complete else 'INCOMPLETE (no manifest: interrupted run)'
    print(f'snapshot {SNAPSHOT_ID} already exists: {state}')
    print(f'content: {sorted(os.listdir(snapshot_dir))}')
    print('Nothing written. Write-once: this notebook never overwrites a snapshot.')
    if not complete:
        print('Interrupted run: delete this folder by hand, then re-run with the same id.')
    notebookutils.fs.unmount(MOUNT_POINT)
    notebookutils.notebook.exit('already-present' if complete else 'incomplete-folder')

print(f'snapshot {SNAPSHOT_ID} does not exist yet: OK to write')

## 2. Read the Delta table at ONE fixed version

The latest version is looked up once, then the table is read `versionAsOf` that version. The number goes into the manifest as `source_delta_version`.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Minutes are written as UTC instants; make the session explicit.
spark.conf.set('spark.sql.session.timeZone', 'UTC')

source_delta_version = int(
    DeltaTable.forPath(spark, ONELAKE_PATH).history(1).select('version').collect()[0][0]
)
raw = spark.read.format('delta').option('versionAsOf', source_delta_version).load(ONELAKE_PATH)
print(f'source_delta_version : {source_delta_version}')

stats = raw.select(
    F.count('*').alias('rows'),
    F.countDistinct('device_id').alias('devices'),
    F.min('timestamp').alias('first_seen'),
    F.max('timestamp').alias('last_seen'),
).collect()[0]

span_min = (stats['last_seen'] - stats['first_seen']).total_seconds() / 60 if stats['rows'] else 0
print(f"rows       : {stats['rows']:,}")
print(f"devices    : {stats['devices']}")
print(f"first_seen : {stats['first_seen']}")
print(f"last_seen  : {stats['last_seen']}")
print(f'span       : {span_min:.0f} min  (~{span_min / 60:.1f} h)')

if stats['rows'] == 0:
    raise RuntimeError(
        'The Delta mirror is empty: mirroring has not caught up yet, or nothing was ingested '
        'since OneLake availability was enabled (Backfill is false). Nothing was written.'
    )

usable_min = max(span_min - (WINDOW_MIN - 1) - HORIZON_MIN, 0)
print(f'usable     : {usable_min:.0f} min per device after warm-up and tail')
if usable_min < RECOMMENDED_MIN_PER_DEVICE:
    print(f'NOTE: below the comfortable {RECOMMENDED_MIN_PER_DEVICE} min. '
          'Replay history with: python simulator/sensor_producer.py --backfill-minutes 1440')

## 3. Feature engineering (identical to `01_snapshot_to_datastore`)

In [ ]:
grid = (
    raw.withColumn('minute', F.date_trunc('minute', F.col('timestamp')))
    .groupBy('device_id', 'minute')
    .agg(*[F.avg(c).alias(c) for c in SENSORS])
)

order = Window.partitionBy('device_id').orderBy('minute')

gaps = (
    grid.withColumn('prev', F.lag('minute').over(order))
    .withColumn('delta_min', (F.col('minute').cast('long') - F.col('prev').cast('long')) / 60)
    .filter(F.col('delta_min') > 1)
)
gap_count = gaps.count()
print(f'minutes on the grid : {grid.count():,}')
print(f'gaps > 1 min        : {gap_count}')
if gap_count:
    gaps.select('device_id', 'minute', 'delta_min').orderBy(F.desc('delta_min')).show(5, False)

roll = order.rowsBetween(-(WINDOW_MIN - 1), 0)

feat = grid
for sensor in SENSORS:
    for lag in LAGS_MIN:
        feat = feat.withColumn(f'{sensor}_lag{lag}', F.lag(sensor, lag).over(order))

for sensor in ['temperature_c', 'vibration_mm_s']:
    feat = feat.withColumn(f'{sensor}_mean{WINDOW_MIN}', F.avg(sensor).over(roll))
    feat = feat.withColumn(f'{sensor}_std{WINDOW_MIN}', F.stddev(sensor).over(roll))

feat = (
    feat.withColumn(
        'temperature_slope15',
        (F.col('temperature_c') - F.col('temperature_c_lag15')) / 15.0,
    )
    .withColumn('hour_of_day', F.hour('minute'))
    .withColumn(TARGET, F.lead('temperature_c', HORIZON_MIN).over(order))
)

dataset = feat.dropna()

feature_columns = [c for c in dataset.columns if c not in ('device_id', 'minute', TARGET)]
print(f'feature rows   : {dataset.count():,}')
print(f'feature columns: {len(feature_columns)}')

per_device = dataset.groupBy('device_id').count().orderBy('device_id')
per_device.show()
min_rows = per_device.agg(F.min('count')).collect()[0][0] or 0
if min_rows < MIN_USABLE_MIN_PER_DEVICE:
    raise RuntimeError(
        f'Thinnest device has only {min_rows} usable minutes (floor {MIN_USABLE_MIN_PER_DEVICE}). '
        'Replay history, wait one mirroring window (5 min), re-run. Nothing was written.'
    )

## 4. Write the snapshot: data first, manifest LAST

In [ ]:
import hashlib
import json


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


pdf = dataset.orderBy('device_id', 'minute').toPandas()

# exist_ok=False: a concurrent run that created the folder meanwhile makes this fail loudly.
os.makedirs(snapshot_dir, exist_ok=False)
pdf.to_parquet(data_path, index=False)
data_sha256 = sha256_file(data_path)
print(f'written: {data_path}')
print(f'shape  : {pdf.shape[0]:,} rows x {pdf.shape[1]} columns')

manifest = {
    'site': SITE,
    'snapshot_id': SNAPSHOT_ID,
    'data_file': DATA_FILE,
    'data_sha256': data_sha256,
    'row_count': int(len(pdf)),
    'window_start': pdf['minute'].min().isoformat(),
    'window_end': pdf['minute'].max().isoformat(),
    'columns': [str(c) for c in pdf.columns],
    'source_table': TABLE_NAME,
    'source_delta_version': source_delta_version,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'horizon_min': HORIZON_MIN,
    'window_min': WINDOW_MIN,
}

# Canonical serialisation: sorted keys, UTF-8, trailing newline.
manifest_bytes = (json.dumps(manifest, sort_keys=True, indent=2, ensure_ascii=False) + '\n').encode('utf-8')
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()

# 'xb': refuses to overwrite. Written after the data on purpose.
with open(manifest_path, 'xb') as handle:
    handle.write(manifest_bytes)
print(f'written: {manifest_path}')

## 5. Read back and report

In [ ]:
# Independent read-back from the mount: what is stored must hash to what was announced.
readback_manifest = sha256_file(manifest_path)
readback_data = sha256_file(data_path)
assert readback_manifest == manifest_sha256, f'manifest hash mismatch {readback_manifest} != {manifest_sha256}'
assert readback_data == data_sha256, f'data hash mismatch {readback_data} != {data_sha256}'
assert sorted(os.listdir(snapshot_dir)) == sorted([DATA_FILE, MANIFEST_FILE]), os.listdir(snapshot_dir)

notebookutils.fs.unmount(MOUNT_POINT)

print('--- SUMMARY (report in docs/PROGRESS.md) ---')
print(f'snapshot_id          : {SNAPSHOT_ID}')
print(f'abfss folder         : {abfss_root}/{SNAPSHOT_PREFIX}/')
print(f'rows                 : {manifest["row_count"]:,}')
print(f'window               : {manifest["window_start"]} -> {manifest["window_end"]}')
print(f'source_delta_version : {source_delta_version}')
print(f'data_sha256          : {data_sha256}')
print(f'manifest_sha256      : {manifest_sha256}')
print('read-back            : OK (hashes and folder content match)')
print()
print('Next: register the data asset')
print(f'  python scripts/72_register_snapshot_asset.py --snapshot-id {SNAPSHOT_ID}')